# LA Studio TTS — Qwen3-TTS CustomVoice 1.7B

This notebook loads exactly `qwen3-tts-1.7b-customvoice` (`Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's TTS panel.


In [ ]:
!nvidia-smi
%pip install -q "qwen-tts==0.1.1" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_tts_worker.py')
WORKER.write_text('import io\nimport os\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TTS_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass SpeechRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice: str = Field(default="auto", max_length=160)\n    language: str = Field(default="auto", max_length=40)\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\n    response_format: str = "wav"\n    settings: dict = Field(default_factory=dict)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef wav_response(samples, sample_rate: int):\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\n    if audio.size == 0:\n        raise RuntimeError("the selected model returned no audio")\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    if not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned non-finite audio")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\nfrom qwen_tts import Qwen3TTSModel\n\nMODEL_ID = "qwen3-tts-1.7b-customvoice"\nMODEL_NAME = "Qwen3-TTS CustomVoice 1.7B"\nUPSTREAM_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"\nSUPPORTED_LANGUAGES = ["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"]\nSUPPORTED_VOICES = ["Aiden", "Dylan", "Eric", "Ono_Anna", "Ryan", "Serena", "Sohee", "Uncle_Fu", "Vivian"]\nMODEL = Qwen3TTSModel.from_pretrained(UPSTREAM_MODEL, device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")\n\ndef qwen_language(value: str):\n    normalized = value.strip().lower()\n    mapping = {"auto": "Auto", "zh": "Chinese", "en": "English", "ja": "Japanese", "ko": "Korean", "de": "German", "fr": "French", "ru": "Russian", "pt": "Portuguese", "es": "Spanish", "it": "Italian"}\n    return mapping.get(normalized, value.strip().title() or "Auto")\n\ndef synthesize_exact_model(request: SpeechRequest):\n    speaker = request.voice.strip() or "Aiden"\n    if speaker.lower() not in {voice.lower() for voice in SUPPORTED_VOICES}:\n        raise HTTPException(status_code=422, detail="unsupported Qwen3 CustomVoice speaker")\n    canonical = next(voice for voice in SUPPORTED_VOICES if voice.lower() == speaker.lower())\n    instruct = str(request.settings.get("instruct", "")).strip()\n    wavs, sample_rate = MODEL.generate_custom_voice(\n        text=request.input,\n        language=qwen_language(request.language),\n        speaker=canonical,\n        instruct=instruct,\n    )\n    return wavs[0], sample_rate\n\napp = FastAPI(title=f"LA Studio TTS — {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "tts",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "voices": SUPPORTED_VOICES,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/speech")\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if request.response_format.strip().lower() != "wav":\n        raise HTTPException(status_code=422, detail="this worker returns WAV audio only")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab TTS worker is busy; retry shortly")\n    try:\n        samples, sample_rate = synthesize_exact_model(request)\n        return wav_response(samples, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'qwen3-tts-1.7b-customvoice'

import os, re, secrets, subprocess, sys, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env["LA_STUDIO_COLAB_TTS_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "la_studio_tts_worker:app", "--host", "127.0.0.1", "--port", "3921"],
    cwd="/content",
    env=env,
)
for _ in range(180):
    try:
        check = urllib.request.Request(
            "http://127.0.0.1:3921/health",
            headers={"Authorization": "Bearer " + TOKEN},
        )
        with urllib.request.urlopen(check, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError("The exact-model TTS worker did not become ready. Inspect the cell output above.")

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3921", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_TTS_URL=" + public_url)
print("LA_STUDIO_COLAB_TTS_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_TTS_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
